# CREST Reaction Network Runner

This notebook provides a CLI-compatible workflow for generating CREST 3.0.2 reaction networks from a seed structure. Each section is split into functional blocks with explanations.


## Imports and CLI parsing

This block defines the argument parser so the notebook can be used like a script (e.g., `python crest_rxn_network.ipynb --seed ...`).


In [ ]:
from __future__ import annotations

import argparse
import json
import os
from pathlib import Path
import shlex
import shutil
import subprocess
from typing import Iterable


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Generate a reaction network using CREST 3.0.2.",
    )
    parser.add_argument(
        "--seed",
        required=True,
        help="Path to the input seed .xyz file.",
    )
    parser.add_argument(
        "--workdir",
        required=True,
        help="Working directory for CREST outputs.",
    )
    parser.add_argument(
        "--charge",
        required=True,
        type=int,
        help="Total molecular charge.",
    )
    parser.add_argument(
        "--uhf",
        required=True,
        type=int,
        help="Number of unpaired electrons (UHF).",
    )
    parser.add_argument(
        "--solvent",
        help="Optional solvent name for ALPB solvation model.",
    )
    parser.add_argument(
        "--nprocs",
        type=int,
        help="Optional number of CPU cores to use.",
    )
    parser.add_argument(
        "--ionization",
        help="Optional ionization metadata (e.g., EI).",
    )
    parser.add_argument(
        "--instrument",
        help="Optional instrument metadata (e.g., GC-MS).",
    )
    return parser.parse_args()


## Validation and utility helpers

These helpers validate the seed file, manage directories, and organize outputs.


In [ ]:
def require_readable_xyz(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Seed file not found: {path}")
    if not path.is_file():
        raise FileNotFoundError(f"Seed path is not a file: {path}")
    if not os.access(path, os.R_OK):
        raise PermissionError(f"Seed file is not readable: {path}")
    if path.suffix.lower() != ".xyz":
        raise ValueError(f"Seed file must be .xyz: {path}")


def ensure_directory(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


def build_crest_command(
    seed_filename: str,
    charge: int,
    uhf: int,
    solvent: str | None,
) -> list[str]:
    # CREST reaction path module flag (reaction network generation).
    cmd = [
        "crest",
        seed_filename,
        "--rpath",
        "-chrg",
        str(charge),
        "-uhf",
        str(uhf),
    ]
    if solvent:
        cmd.extend(["--alpb", solvent])
    return cmd


def set_thread_env(base_env: dict[str, str], nprocs: int | None) -> dict[str, str]:
    env = base_env.copy()
    if nprocs:
        for key in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"):
            env[key] = str(nprocs)
    return env


def move_outputs(
    workdir: Path,
    output_dir: Path,
    keep: Iterable[Path],
) -> None:
    keep_set = {path.resolve() for path in keep}
    for item in workdir.iterdir():
        if item.resolve() in keep_set:
            continue
        if item.resolve() == output_dir.resolve():
            continue
        target = output_dir / item.name
        if target.exists():
            if target.is_dir():
                shutil.rmtree(target)
            else:
                target.unlink()
        shutil.move(str(item), str(target))


def relative_paths(paths: Iterable[Path], base: Path) -> list[str]:
    rel_paths = []
    for path in paths:
        try:
            rel_paths.append(str(path.relative_to(base)))
        except ValueError:
            rel_paths.append(str(path))
    return sorted(rel_paths)


def find_matching(output_dir: Path, patterns: Iterable[str]) -> list[Path]:
    matches: list[Path] = []
    for pattern in patterns:
        matches.extend(output_dir.glob(pattern))
    return sorted({path for path in matches})


## Main execution flow

This block validates inputs, runs CREST, captures logs, organizes outputs, and writes the manifest with EI/GC-MS metadata.


In [ ]:
def main() -> None:
    args = parse_args()
    seed_path = Path(args.seed).expanduser().resolve()
    workdir = Path(args.workdir).expanduser().resolve()

    require_readable_xyz(seed_path)
    ensure_directory(workdir)

    seed_copy = workdir / seed_path.name
    shutil.copy2(seed_path, seed_copy)

    crest_log = workdir / "crest.log"
    crest_cmd = build_crest_command(
        seed_copy.name,
        args.charge,
        args.uhf,
        args.solvent,
    )
    env = set_thread_env(os.environ, args.nprocs)

    with crest_log.open("w", encoding="utf-8") as log_file:
        result = subprocess.run(
            crest_cmd,
            cwd=workdir,
            env=env,
            stdout=log_file,
            stderr=subprocess.STDOUT,
            check=False,
        )

    if result.returncode != 0:
        raise RuntimeError(
            "CREST reaction network generation failed. "
            f"See log for details: {crest_log}"
        )

    output_dir = workdir / "output"
    ensure_directory(output_dir)
    move_outputs(
        workdir,
        output_dir,
        keep=[seed_copy, crest_log, output_dir],
    )

    network_files = find_matching(
        output_dir,
        ["*rxn*", "*network*", "*.rxn"],
    )
    structure_sets = find_matching(
        output_dir,
        ["*.xyz", "*.sdf", "*.mol"],
    )

    manifest = {
        "seed_input": str(seed_path),
        "seed_copy": str(seed_copy),
        "workdir": str(workdir),
        "crest_command": {
            "argv": crest_cmd,
            "shell": shlex.join(crest_cmd),
        },
        "crest_log": str(crest_log),
        "output_dir": str(output_dir),
        "network_files": relative_paths(network_files, workdir),
        "structure_sets": relative_paths(structure_sets, workdir),
        "metadata": {
            "charge": args.charge,
            "uhf": args.uhf,
            "solvent": args.solvent,
            "nprocs": args.nprocs,
            "ionization": args.ionization,
            "instrument": args.instrument,
        },
    }

    manifest_path = workdir / "network_manifest.json"
    with manifest_path.open("w", encoding="utf-8") as handle:
        json.dump(manifest, handle, indent=2, ensure_ascii=False)

    network_summary = ", ".join(manifest["network_files"]) or "(none detected)"
    print("CREST reaction network completed.")
    print(f"Output directory: {output_dir}")
    print(f"Network files: {network_summary}")
    print(f"Log file: {crest_log}")


if __name__ == "__main__":
    main()


## Usage notes

Run this notebook as a script via a Jupyter-aware runner, for example:

```bash
python scripts/crest_rxn_network.ipynb --seed path/to/seed.xyz --workdir /tmp/run \
  --charge 0 --uhf 0 --solvent water --nprocs 8 --ionization EI --instrument GC-MS
```
